# 5.4 - Context Engineering

**Module 5 - Prompt Engineering** | Netsetos GenAI Engineering

This notebook treats the context window as a finite, attention-limited token budget and builds a small toolkit for spending it well: measure the budget, order the pieces, retrieve instead of stuff, compact what grows, and assemble it all into one pipeline. Every technique is a real function you can run against Gemini through the current unified `google-genai` SDK.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Standard environment prep. We import NumPy (used later for the embedding-similarity math) and pin the random seed so any seeded values are reproducible across runs.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

A one-line environment cell, not a model call. It just makes NumPy available and fixes the RNG seed so the notebook behaves the same every time you run it.

**How the code works - step by step**
- **`# !pip install numpy -q`** - commented install for a fresh environment such as Colab; uncomment if NumPy is missing.
- **`import numpy as np`** - the array/linear-algebra library used by the cosine-similarity math in Step 4.
- **`np.random.seed(42)`** - fixes the random seed so any seeded values are reproducible.

**In one line:** import NumPy and lock the seed for reproducibility.

**What you'll see:** (no output - environment prep)

## 1 - The two helpers you'll reuse: `ask()` and `count_tokens()`

Every example in this notebook either sends a completion or measures a token budget, so we define both once. `ask()` is a thin wrapper over a Gemini completion; `count_tokens()` calls the real token-counting endpoint - because you cannot manage a budget you never measured.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# pip install "google-genai>=1.0.0"
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-3-flash"

def ask(prompt: str, temperature: float = 0.2, system: str = None) -> str:
    cfg = types.GenerateContentConfig(temperature=temperature, system_instruction=system)
    try:
        return client.models.generate_content(model=MODEL, contents=prompt, config=cfg).text.strip()
    except Exception as e:
        return f"[API error: {e}]"   # teaching-only: real code raises/retries - never let an error string become a summary or a cached prefix

def count_tokens(text: str) -> int:
    """Measure tokens BEFORE sending - you can't manage what you don't measure."""
    return client.models.count_tokens(model=MODEL, contents=text).total_tokens

print(count_tokens("How many tokens is this sentence?"))
# Output: 8

**Explanation**

This is the shared toolbox cell, built on the current unified `google-genai` SDK (the older `google.generativeai` package was deprecated in 2025). One reusable client backs two tiny helpers: one to get an answer, one to weigh text against the budget. Everything downstream calls these two functions.

**How the code works - step by step**
- **`genai.Client(...)`** - creates one reusable client from the `GOOGLE_API_KEY` env var, instead of the deprecated global `configure()` + per-call `GenerativeModel` pattern.
- **`MODEL = "gemini-3-flash"`** - the completion model used by `ask`.
- **`ask(prompt, temperature, system)`** - builds a `GenerateContentConfig`, calls `generate_content`, and returns the stripped text; the `try/except` returns an `[API error: ...]` string so one network blip can't crash a multi-call pipeline (a teaching shortcut - real code retries).
- **`count_tokens(text)`** - calls the real `client.models.count_tokens` endpoint to measure a piece of text before you send it.
- **`print(count_tokens(...))`** - a smoke test that the client works and returns a token count.

**In one line:** one client, an `ask` completion helper, and an honest `count_tokens` measure.

**What you'll see:** Prints `8` - the token count of the test sentence "How many tokens is this sentence?", confirming the client and token endpoint are live.

## 2 - Build a token budget for a request

The context window is a fixed wallet paid in tokens, shared by the system prompt, examples, history, retrieved docs, and the answer. This cell sums the cost of every part and checks whether anything is left over for the model to actually respond with.

> **Needs a Gemini API key** - `fits_budget` calls `count_tokens` on each part.

In [ ]:
def fits_budget(parts: dict, window: int = 200_000, reserve_out: int = 2_000) -> dict:
    """Sum the token cost of every context part; check it leaves room for the answer."""
    used = {name: count_tokens(text) for name, text in parts.items()}
    total = sum(used.values())
    free = window - total - reserve_out
    return {"per_part": used, "total": total, "free_for_answer": free, "fits": free >= 0}

# Inline sample context (in a real app these come from files / a DB / the live conversation):
ctx = {
    "system": "You are a support assistant.",                                   # small, fixed
    "examples": "Q: How do I get a refund?\nA: Return within 7 days.\n" * 50,     # medium, fixed
    "history": "user: my order is late\nassistant: let me check that\n" * 200,   # grows every turn
    "retrieved": "Refund policy: defective items return within 7 days.\n" * 500, # varies per query
}
print(fits_budget(ctx))
# Output (illustrative): {'per_part': {'system': 6, 'examples': 7900, 'history': 31200,
#          'retrieved': 162000}, 'total': 201106, 'free_for_answer': -3106, 'fits': False}

**Explanation**

A measurement harness, not a model call. `fits_budget` counts each context component separately so you can see which one is the budget hog, then subtracts the total plus a reserved output allowance from the window to decide whether the request fits. It embodies the "measure before you manage" habit the rest of the lesson depends on.

**How the code works - step by step**
- **`used = {name: count_tokens(text) ...}`** - counts every part on its own, so the per-component cost is visible (here the retrieved docs dominate).
- **`total = sum(used.values())`** - the tokens the context alone consumes.
- **`free = window - total - reserve_out`** - subtracts a reserved output allowance (`reserve_out`) up front, because a window full to the brim leaves nothing to answer with.
- **`return {... 'fits': free >= 0}`** - `free_for_answer` goes negative and `fits` turns False the moment you overflow - the signal to trim or retrieve less before sending.
- **`ctx = {...}` then `print(fits_budget(ctx))`** - runs the check on a sample context mixing fixed, growing, and per-query parts.

**In one line:** count each part -> reserve output space -> see if the answer still fits.

**What you'll see:** Prints a dict with `per_part` token counts, a `total` of ~201k, a negative `free_for_answer` (~-3106), and `fits: False` - the sample context overflows a 200k window (values are illustrative).

## 3 - Order the context so signal lands where the model attends

Since attention is position-dependent (strong at the edges, weak in the middle), ordering is a lever. This cell assembles a prompt like a good briefing: the instruction leads, bulk reference sits in the low-attention middle, and the actual question comes last.

In [ ]:
def build_context(task: str, docs: list[str], question: str) -> str:
    """Signal at the edges, bulk reference in the middle, fenced sections."""
    docs_block = "\n\n".join(f"<doc id={i}>\n{d}\n</doc>" for i, d in enumerate(docs))
    return f"""# TASK (read this first)
{task}

# REFERENCE MATERIAL
{docs_block}

# QUESTION (answer this, using only the reference above)
{question}"""     # the ask sits LAST - recency works for you

prompt = build_context(
    task="Answer strictly from the reference. If it is not there, say 'not found'.",
    docs=["Refund policy: defective items may be returned within 7 days of delivery for a full refund.",
          "...policy B...", "...policy C..."],
    question="What is the refund window for a defective item?")
print(ask(prompt))
# Output: 7 days from delivery for a defective item (per doc 1). [grounded, on-task]

**Explanation**

A prompt-assembly function that turns loose pieces into a structured, position-aware prompt. The core move is placement: high-signal content (the task and the ask) goes at the edges, bulk documents go in the middle, and every document is fenced into a labelled section so the model can tell them apart and cite by id.

**How the code works - step by step**
- **`docs_block = ...<doc id=i>...`** - wraps each reference doc in a fenced, id-tagged section so the model can distinguish and cite them, and can't mistake a doc for an instruction.
- **`# TASK (read this first)`** - the instruction and the grounding rule lead, landing in a high-attention position.
- **`# REFERENCE MATERIAL`** - the bulk docs sit in the middle, the lowest-attention zone, which is right for material to consult rather than obey.
- **`# QUESTION ...`** - the ask sits last, exploiting recency so it's freshest when generation starts.
- **`prompt = build_context(...)` then `ask(prompt)`** - builds the ordered prompt over three sample docs and sends it.

**In one line:** instruction at the top, docs in the middle, question at the bottom, all fenced.

**What you'll see:** Prints a grounded, on-task answer - roughly "7 days from delivery for a defective item," cited to the relevant doc - showing the ordered prompt kept the model on target.

## 4 - Retrieve the relevant few, then reason over them

Instead of pasting a whole corpus into the window, embed the query, find the documents whose vectors sit nearest, and send only those. This is the 2026 hybrid default: retrieve a handful of high-signal tokens, then long-context-reason over them.

> **Needs a Gemini API key** - `embed` calls the embeddings endpoint for the query and every doc.

In [ ]:
import numpy as np

def embed(text: str) -> np.ndarray:
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

def retrieve(query: str, corpus: list[str], k: int = 3) -> list[str]:
    """Return the k documents nearest the query. DEMO ONLY: this re-embeds every
    doc on each call. In production, embed the corpus ONCE into a vector DB
    (Module 8) - never embed on the request path."""
    q = embed(query)
    qn = np.linalg.norm(q)
    scored = []
    for d in corpus:
        e = embed(d)                                   # embed each doc once, not twice
        scored.append((float(np.dot(q, e) / (qn * np.linalg.norm(e))), d))
    scored.sort(reverse=True)
    return [d for _, d in scored[:k]]

big_corpus = [
    "Refund policy: defective items may be returned within 7 days of delivery for a full refund.",
    "Shipping policy: standard delivery is 3-5 business days.",
    "Warranty: electronics carry a 1-year limited warranty.",
]

question = "What is the refund window for a defective item?"
top = retrieve(question, big_corpus, k=3)         # 3 of 500, not all 500
answer = ask(build_context("Answer only from the reference.", top, question))
print(answer)
# Output: 7 days from delivery for a defective item. [from the 3 retrieved docs]

**Explanation**

A minimal, from-scratch retrieval step wired to the ordering step from before. `embed` turns text into a vector; `retrieve` scores every doc against the query by cosine similarity and keeps the top k; then the winners feed the same `build_context` so retrieval and long-context work as partners. It's deliberately a demo - it re-embeds the corpus on every call, which production never does.

**How the code works - step by step**
- **`embed(text)`** - calls `gemini-embedding-001` and returns the embedding as a NumPy array.
- **`retrieve(query, corpus, k)`** - embeds the query, then embeds each doc and scores it by cosine similarity (`dot(q,e) / (|q||e|)`), sorts, and returns the top k. The docstring flags that production embeds the corpus once into a vector DB, never on the request path.
- **`big_corpus = [...]`** - three sample policy docs to search over.
- **`top = retrieve(question, big_corpus, k=3)`** - selects the relevant subset (here 3 of many).
- **`answer = ask(build_context(...))`** - feeds only the retrieved docs into the ordered prompt from Step 3 and answers.

**In one line:** embed and rank by cosine similarity -> keep the top k -> reason over just those.

**What you'll see:** Prints a grounded answer ("7 days from delivery for a defective item") derived only from the retrieved docs - the model saw the relevant few, not the whole corpus.

## 5 - Compact a growing conversation

Chat history grows every turn and eventually blows the budget. This cell keeps the system prompt and the most recent turns verbatim, and summarizes the older middle into a few bullet facts - shrinking the token count without losing the thread.

> **Needs a Gemini API key** - the summary is produced by an `ask` call.

In [ ]:
def compact_history(system: str, turns: list[str], keep_recent: int = 6) -> str:
    """Keep system + the last N turns verbatim; summarize the older middle."""
    if len(turns) <= keep_recent:
        return system + "\n" + "\n".join(turns)
    old, recent = turns[:-keep_recent], turns[-keep_recent:]
    summary = ask("Summarize these earlier turns into 5 bullet facts to remember "
                  "(ids, names, decisions). Keep it under 120 words:\n" + "\n".join(old))
    return f"{system}\n\n# EARLIER (summary)\n{summary}\n\n# RECENT\n" + "\n".join(recent)

all_turns = [f"turn {i}: customer and agent exchange about order #{1000 + i}" for i in range(20)]
ctx = compact_history("You are a support agent.", all_turns, keep_recent=6)
print(count_tokens(ctx))
# Output: 1840          (was 31200 raw - a ~17x reduction, thread preserved)
# Note: summarizing is not free - it adds a model round-trip (latency + tokens) and another
# point that can fail, so cache the summary and recompact periodically, not every turn.

**Explanation**

A history-compaction function that trades a little fidelity on old turns for a large budget recovery. The privileged parts (system prompt, recent turns) stay exact because they sit in high-attention positions and carry the live thread; the old middle is compressed rather than deleted, so nothing load-bearing is silently dropped.

**How the code works - step by step**
- **`if len(turns) <= keep_recent`** - short conversations are returned untouched; no summarization needed.
- **`old, recent = turns[:-keep_recent], turns[-keep_recent:]`** - splits off the last N turns to keep verbatim from the older middle to compress.
- **`summary = ask("Summarize these earlier turns into 5 bullet facts...")`** - one model round-trip that distills the old turns into ids, names, and decisions under 120 words.
- **`return f"{system}... # EARLIER (summary) ... # RECENT ..."`** - reassembles system prompt + summary + recent turns into a fenced, compact context.
- **`ctx = compact_history(...)` then `print(count_tokens(ctx))`** - compacts a 20-turn history and measures the result.

**In one line:** keep system + recent turns exact, summarize the old middle, reassemble.

**What you'll see:** Prints a small token count (~1840 vs ~31200 raw, roughly a 17x reduction), showing the history was compressed while the thread was preserved.

## 6 - The whole discipline in one pipeline

This final cell chains every technique into a single function: retrieve the relevant docs, compact the history, order the pieces, check the budget, and answer - with a fallback that retrieves *less* if it overflows rather than truncating blindly.

> **Needs a Gemini API key** - it calls `retrieve`, `compact_history`, `fits_budget`, and `ask` end to end.

In [ ]:
def answer_with_context_engineering(question: str, corpus: list[str],
                                   history: list[str], system: str) -> str:
    """The whole lesson in one pipeline: retrieve -> compact -> order -> budget -> answer."""
    docs = retrieve(question, corpus, k=3)                  # Step 4: don't stuff
    convo = compact_history(system, history, keep_recent=6)     # Step 5: compress old turns
    prompt = build_context(convo, docs, question)               # Step 3: order it
    budget = fits_budget({"prompt": prompt})                    # Step 1: check it fits
    if not budget["fits"]:
        docs = retrieve(question, corpus, k=2)              # over budget? retrieve less
        prompt = build_context(convo, docs, question)
    return ask(prompt)

# Inputs for the demo (big_corpus reuses the small doc list from Concept 4):
q = "What is the refund window for a defective item?"
big_corpus = [
    "Refund policy: defective items may be returned within 7 days of delivery for a full refund.",
    "Shipping policy: standard delivery is 3-5 business days.",
    "Warranty: electronics carry a 1-year limited warranty.",
]
chat_history = [f"turn {i}: customer and agent exchange about order #{1000 + i}" for i in range(10)]
system_prompt = "You are a support agent. Answer strictly from the reference."

print(answer_with_context_engineering(q, big_corpus, chat_history, system_prompt))
# Output: a grounded, on-task answer from a context that fits and stays high-signal

**Explanation**

The capstone cell that composes Steps 1, 3, 4, and 5 into one context-engineering pipeline. Read it as the discipline as a system, not a single prompt: each helper you built handles one job, and the budget check closes the loop by shrinking retrieval when the assembled context won't fit.

**How the code works - step by step**
- **`docs = retrieve(question, corpus, k=3)`** - Step 4: pull the few relevant docs so the context starts high-signal.
- **`convo = compact_history(system, history, keep_recent=6)`** - Step 5: shrink the chat history without losing the thread.
- **`prompt = build_context(convo, docs, question)`** - Step 3: order task, docs, and question where the model attends best.
- **`budget = fits_budget({"prompt": prompt})`** - Step 1: check it fits and reserve answer space.
- **`if not budget["fits"]: docs = retrieve(..., k=2)`** - on overflow, retrieve fewer docs and rebuild rather than truncating blindly.
- **`return ask(prompt)`** - send the assembled, budgeted context.
- **`q`, `big_corpus`, `chat_history`, `system_prompt`** then **`print(answer_with_context_engineering(...))`** - runs the full pipeline on a demo request.

**In one line:** retrieve -> compact -> order -> budget (retrieve less if over) -> answer.

**What you'll see:** Prints a grounded, on-task answer produced from a context that was retrieved, compacted, ordered, and confirmed to fit - the whole lesson working as one system.

You now have the context discipline as runnable parts: `count_tokens`/`fits_budget` measure and reserve the budget, `build_context` orders signal to the edges, `retrieve` sends only the relevant few, `compact_history` shrinks a growing conversation, and `answer_with_context_engineering` chains all four in the order retrieve -> compact -> order -> budget. The through-line: the smallest high-signal context wins, because a clean small context has no middle to get lost in. From here, Lesson 5.5 adds structured output, Module 8 builds the full retrieval stack you faked with `retrieve`, and Module 11 grows the compaction and memory ideas into real agent memory.